# ⚽ Premier League — Predictor de Próximo Partido

Pipeline completo de Machine Learning que, dado el nombre de un equipo de la Premier League:
1. Identifica su **próximo partido** (vía API football-data.org)
2. Calcula **características avanzadas** (Elo, ventanas rolling, H2H, fatiga, forma al descanso…)
3. Entrena el **mejor modelo posible** (Random Forest vs XGBoost con GridSearchCV)
4. Devuelve las **probabilidades exactas** de Victoria / Empate / Derrota

---
**Fuentes de datos:**
- 📂 *Local*: Excel con partidos desde 2016/17 hasta hoy (`data/`)
- 🌐 *API*: football-data.org (estadísticas de tiros, posesión, etc.)

## 📦 Celda 1 — Imports y configuración global

In [1]:
# ─── IMPORTS ────────────────────────────────────────────────────────────────
import os
import glob
import json
import time
import warnings
import numpy as np
import pandas as pd
import requests
from datetime import datetime, timedelta
from pathlib import Path

# ML
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, log_loss, classification_report
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

# ─── CONFIGURACIÓN ──────────────────────────────────────────────────────────
# ⚠️  Introduce aquí tu API key de football-data.org
API_KEY  = "da77f8cabbc54b4caa2486cc4418377b"  # <── REEMPLAZA CON TU KEY
BASE_URL = "https://api.football-data.org/v4"
COMP_CODE = "PL"               # Premier League
RATE_LIMIT_SLEEP = 6.2         # segundos entre peticiones (10 req/min máx.)
CACHE_FILE = Path("api_cache.json")

# Ruta a los datos históricos locales
DATA_DIR  = Path("data")

# Equipo a predecir (modifica este valor o cámbialo en la celda final)
EQUIPO_OBJETIVO = "Arsenal FC"

print("✅ Imports y configuración cargados.")
print(f"   Equipo objetivo: {EQUIPO_OBJETIVO}")
print(f"   DATA_DIR:        {DATA_DIR.resolve()}")

✅ Imports y configuración cargados.
   Equipo objetivo: Arsenal FC
   DATA_DIR:        C:\Users\Mikel\Desktop\PROYECTOS\Next Match Predictor\data


## 📂 Celda 2 — Cliente de la API con caché

In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CLIENTE API con caché en disco y control de rate-limit
# CORRECCIONES APLICADAS:
#  1. Timezone-aware: pd.to_datetime(..., utc=True) + pd.Timestamp.now(tz='UTC')
#     FIX del error: "can't compare offset-naive and offset-aware datetimes"
#  2. Fallback a /teams/{id}/matches?status=SCHEDULED si la temporada no tiene futuros
#  3. Mensajes de error explícitos para 403/400/429 con el body de la respuesta
#  4. Método clear_cache() para forzar datos frescos
#  5. Verificación automática de conexión al instanciar (test de get_teams)
#  6. season=None usa 'if season is not None' en vez de 'if season' (evita season=0)
# ─────────────────────────────────────────────────────────────────────────────

class ApiClient:
    """
    Wrapper de football-data.org v4 con:
    - Caché en disco (api_cache.json) para no repetir peticiones
    - Sleep automático para respetar el límite de 10 req/min
    - Reintentos con backoff en caso de error 429
    """

    def __init__(self, api_key: str, cache_file: Path = CACHE_FILE):
        self.headers   = {"X-Auth-Token": api_key}
        self.cache_file = cache_file
        self._load_cache()
        self._last_request_time = 0.0

    # ── Caché ────────────────────────────────────────────────────────────────
    def _load_cache(self):
        if self.cache_file.exists():
            with open(self.cache_file, 'r', encoding='utf-8') as f:
                self._cache = json.load(f)
        else:
            self._cache = {}

    def _save_cache(self):
        with open(self.cache_file, 'w', encoding='utf-8') as f:
            json.dump(self._cache, f, ensure_ascii=False, indent=2)

    def clear_cache(self):
        """Limpia la caché en memoria y en disco. Útil para forzar datos frescos."""
        self._cache = {}
        if self.cache_file.exists():
            self.cache_file.unlink()
        print("   🗑️  Caché limpiada.")

    # ── Petición base ────────────────────────────────────────────────────────
    def _get(self, endpoint: str, params: dict = None, cache_ttl_hours: float = 1.0) -> dict:
        """
        Realiza una petición GET con caché.
        cache_ttl_hours=0   → nunca cachea
        cache_ttl_hours=-1  → caché permanente (sin expiración)
        """
        cache_key = f"{endpoint}|{json.dumps(params, sort_keys=True)}"

        # Revisar caché
        if cache_key in self._cache and cache_ttl_hours != 0:
            entry = self._cache[cache_key]
            age_h = (time.time() - entry['ts']) / 3600
            if cache_ttl_hours < 0 or age_h < cache_ttl_hours:
                return entry['data']

        # Rate-limit: respetar 10 req/min
        elapsed = time.time() - self._last_request_time
        if elapsed < RATE_LIMIT_SLEEP:
            time.sleep(RATE_LIMIT_SLEEP - elapsed)

        url = f"{BASE_URL}{endpoint}"
        for attempt in range(3):
            try:
                resp = requests.get(url, headers=self.headers, params=params, timeout=20)
            except requests.RequestException as exc:
                print(f"   ❌ Error de conexión en {endpoint}: {exc}")
                return {}

            self._last_request_time = time.time()

            if resp.status_code == 200:
                data = resp.json()
                if cache_ttl_hours != 0:
                    self._cache[cache_key] = {'ts': time.time(), 'data': data}
                    self._save_cache()
                return data

            elif resp.status_code == 429:
                wait = 65
                print(f"   ⏳ Rate-limit (429). Esperando {wait}s…")
                time.sleep(wait)
                # Reintentar sin break

            elif resp.status_code == 403:
                print(f"   ❌ HTTP 403 Forbidden en {endpoint}")
                print(f"      → Comprueba que la API key es correcta y está activa.")
                print(f"      → Respuesta: {resp.text[:300]}")
                return {}

            elif resp.status_code == 400:
                print(f"   ❌ HTTP 400 Bad Request en {endpoint}")
                print(f"      → Parámetros enviados: {params}")
                print(f"      → Respuesta: {resp.text[:300]}")
                return {}

            else:
                print(f"   ⚠️  HTTP {resp.status_code} en {endpoint}")
                print(f"      Respuesta: {resp.text[:300]}")
                return {}

        print(f"   ❌ Se agotaron los reintentos para {endpoint}")
        return {}

    # ── Endpoints públicos ───────────────────────────────────────────────────
    def get_teams(self) -> pd.DataFrame:
        """
        Lista todos los equipos de la PL de la temporada actual.
        Endpoint: GET /v4/competitions/PL/teams
        Respuesta: { count, filters, competition, season, teams: [{id, name, shortName, tla, ...}] }
        """
        data = self._get(f"/competitions/{COMP_CODE}/teams", cache_ttl_hours=24)
        if not isinstance(data, dict) or not data:
            return pd.DataFrame(columns=['id', 'name', 'shortName', 'tla'])

        teams = data.get('teams', [])
        if not teams:
            return pd.DataFrame(columns=['id', 'name', 'shortName', 'tla'])

        rows = []
        for t in teams:
            if not isinstance(t, dict):
                continue
            rows.append({
                'id':        t.get('id'),
                'name':      t.get('name', ''),
                'shortName': t.get('shortName', ''),
                'tla':       t.get('tla', ''),
            })

        return pd.DataFrame(rows, columns=['id', 'name', 'shortName', 'tla'])

    def find_team_id(self, team_name: str) -> tuple:
        """
        Mapea un nombre de equipo (aproximado) al ID y nombre canónico de la API.
        Estrategia: exacto → normalizado sin 'FC' → primer token.
        """
        df = self.get_teams()
        if df.empty:
            raise ValueError(
                "No se pudieron cargar los equipos desde la API. "
                "Comprueba que la API key es correcta y que tienes conexión a Internet."
            )

        name_lower = team_name.lower().replace(' fc', '').strip()

        # Fase 1 — Coincidencia exacta (normalizada)
        for col in ['name', 'shortName']:
            if col not in df.columns:
                continue
            normalized = (
                df[col].fillna('').astype(str)
                .str.lower()
                .str.replace(' fc', '', regex=False)
                .str.strip()
            )
            mask = normalized == name_lower
            if mask.any():
                row = df.loc[mask].iloc[0]
                return int(row['id']), str(row['name'])

        # Fase 2 — Primer token (ej: 'Arsenal' de 'Arsenal FC')
        first_token = name_lower.split()[0]
        for col in ['name', 'shortName']:
            if col not in df.columns:
                continue
            mask = (
                df[col].fillna('').astype(str)
                .str.lower()
                .str.contains(first_token, regex=False, na=False)
            )
            if mask.any():
                row = df.loc[mask].iloc[0]
                return int(row['id']), str(row['name'])

        available = df['name'].dropna().astype(str).tolist()
        raise ValueError(
            f"Equipo '{team_name}' no encontrado en la API.\n"
            f"Equipos disponibles:\n" + "\n".join(available)
        )

    def get_matches_season(self, season: int = None, cache_ttl_hours: float = 2.0) -> pd.DataFrame:
        """
        Descarga TODOS los partidos de la PL de una temporada en un solo bloque.
        Endpoint: GET /v4/competitions/PL/matches[?season=YYYY]

        IMPORTANTE — Valores correctos del parámetro 'season':
          season=None → temporada actual (la API devuelve la más reciente)
          season=2024 → temporada 2024/25 (el año corresponde al inicio de la temporada)
          season=2025 → temporada 2025/26
          season=2026 → temporada 2026/27

        Estructura de respuesta:
          { count, filters, competition, season,
            matches: [{ id, utcDate, status, matchday,
                        homeTeam: {id, name}, awayTeam: {id, name},
                        score: { winner, duration,
                                 fullTime: {home, away},
                                 halfTime: {home, away} } }] }
        """
        params = {}
        if season is not None:   # FIX: 'is not None' en vez de 'if season' para evitar season=0
            params['season'] = season

        data = self._get(
            f"/competitions/{COMP_CODE}/matches",
            params=params if params else None,
            cache_ttl_hours=cache_ttl_hours
        )
        if not data:
            return pd.DataFrame()

        matches = data.get('matches', [])
        if not matches:
            return pd.DataFrame()

        rows = []
        for m in matches:
            score = m.get('score', {})
            # En la API v4: score.halfTime y score.fullTime → {home: int, away: int}
            ht = score.get('halfTime', {}) or {}
            ft = score.get('fullTime', {}) or {}
            rows.append({
                'match_id':       m.get('id'),
                'utcDate':        m.get('utcDate'),
                'status':         m.get('status'),
                'matchday':       m.get('matchday'),
                'home_id':        m['homeTeam']['id'],
                'home_name':      m['homeTeam']['name'],
                'away_id':        m['awayTeam']['id'],
                'away_name':      m['awayTeam']['name'],
                'goals_home_ft':  ft.get('home'),
                'goals_away_ft':  ft.get('away'),
                'goals_home_ht':  ht.get('home'),
                'goals_away_ht':  ht.get('away'),
                # winner: 'HOME_TEAM' / 'AWAY_TEAM' / 'DRAW' / None
                'winner':         score.get('winner'),
            })

        df = pd.DataFrame(rows)
        if not df.empty:
            # FIX: utc=True genera timestamps timezone-aware (UTC)
            # Esto permite comparar con pd.Timestamp.now(tz='UTC') sin errores
            df['utcDate'] = pd.to_datetime(df['utcDate'], utc=True)
        return df

    def get_next_match(self, team_id: int) -> dict | None:
        """
        Busca el próximo partido programado del equipo en la PL.

        Estrategia de 2 peticiones máximo:
          1. Descarga la temporada actual en bloque y filtra partidos futuros.
          2. Si no hay futuros (fin de temporada), usa /teams/{id}/matches?status=SCHEDULED.
        """
        # Descargar temporada actual en bloque (season=None = la más reciente)
        df = self.get_matches_season(season=None, cache_ttl_hours=1)

        if not df.empty:
            # FIX: pd.Timestamp.now(tz='UTC') es timezone-aware → compatible con df['utcDate']
            # El error "can't compare offset-naive and offset-aware" se producía aquí
            # porque pd.Timestamp.utcnow() NO tiene tz mientras df['utcDate'] SÍ (utc=True)
            now_utc = pd.Timestamp.now(tz='UTC')

            future = df[
                (df['status'].isin(['SCHEDULED', 'TIMED'])) &
                (df['utcDate'] > now_utc) &
                ((df['home_id'] == team_id) | (df['away_id'] == team_id))
            ].sort_values('utcDate')

            if not future.empty:
                return future.iloc[0].to_dict()

        # Fallback: endpoint directo /teams/{id}/matches
        print("   ℹ️  No hay partidos futuros en la temporada actual.")
        print(f"      Consultando /teams/{team_id}/matches?status=SCHEDULED…")
        data = self._get(
            f"/teams/{team_id}/matches",
            params={'status': 'SCHEDULED', 'limit': 5},
            cache_ttl_hours=1
        )
        matches = data.get('matches', [])

        # Filtrar solo partidos de la PL por competition.code
        pl_matches = [
            m for m in matches
            if m.get('competition', {}).get('code') == COMP_CODE
        ]
        if not pl_matches:
            pl_matches = matches  # Si no hay filtro de competición, usar el primero

        if not pl_matches:
            return None

        m = pl_matches[0]
        score = m.get('score', {})
        ht = score.get('halfTime', {}) or {}
        ft = score.get('fullTime', {}) or {}
        return {
            'match_id':       m.get('id'),
            'utcDate':        pd.to_datetime(m.get('utcDate'), utc=True),
            'status':         m.get('status'),
            'matchday':       m.get('matchday'),
            'home_id':        m['homeTeam']['id'],
            'home_name':      m['homeTeam']['name'],
            'away_id':        m['awayTeam']['id'],
            'away_name':      m['awayTeam']['name'],
            'goals_home_ft':  ft.get('home'),
            'goals_away_ft':  ft.get('away'),
            'goals_home_ht':  ht.get('home'),
            'goals_away_ht':  ht.get('away'),
            'winner':         score.get('winner'),
        }

    def get_match_stats(self, match_id: int) -> dict:
        """
        Estadísticas detalladas de un partido (tiros, posesión, etc.).
        Endpoint: GET /v4/matches/{id}
        Nota: Tiros/posesión solo disponibles en planes de pago superiores al gratuito.
        """
        return self._get(f"/matches/{match_id}", cache_ttl_hours=-1)

    def get_team_recent_matches(self, team_id: int, limit: int = 15) -> pd.DataFrame:
        """
        Últimos `limit` partidos FINISHED del equipo en la temporada actual.
        Usa la descarga en bloque (0 peticiones extra si ya está cacheada).
        """
        df = self.get_matches_season(season=None, cache_ttl_hours=2)
        if df.empty:
            return pd.DataFrame()

        finished = df[
            (df['status'] == 'FINISHED') &
            ((df['home_id'] == team_id) | (df['away_id'] == team_id))
        ].sort_values('utcDate', ascending=False).head(limit)

        return finished.reset_index(drop=True)


# ── Instanciar y verificar el cliente ────────────────────────────────────────
api = ApiClient(api_key=API_KEY)

print("⏳ Verificando conexión con la API…")
teams_df = api.get_teams()
if teams_df.empty:
    print("❌ No se pudieron cargar equipos. Revisa la API key y la conexión.")
else:
    print(f"✅ API OK — {len(teams_df)} equipos cargados en la Premier League:")
    for _, row in teams_df.head(5).iterrows():
        print(f"   ID {int(row['id']):5d} | {row['name']}")
    print(f"   … y {len(teams_df)-5} más.")
print(f"   Caché en disco: {CACHE_FILE}")


⏳ Verificando conexión con la API…
✅ API OK — 20 equipos cargados en la Premier League:
   ID    57 | Arsenal FC
   ID    58 | Aston Villa FC
   ID    61 | Chelsea FC
   ID    62 | Everton FC
   ID    63 | Fulham FC
   … y 15 más.
   Caché en disco: api_cache.json


## 📊 Celda 3 — Carga y unión de datos históricos locales

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CARGA DE EXCELS HISTÓRICOS
# ─────────────────────────────────────────────────────────────────────────────

def load_historical_data(data_dir: Path) -> pd.DataFrame:
    """Lee todos los Excel de las subcarpetas de data/ y los une en un único DF."""
    dfs = []
    for xlsx_path in sorted(data_dir.glob("**/*.xlsx")):
        season = xlsx_path.parent.name
        df = pd.read_excel(xlsx_path)
        df['Temporada'] = season
        dfs.append(df)

    if not dfs:
        raise FileNotFoundError(f"No se encontraron archivos .xlsx en {data_dir}")

    df_all = pd.concat(dfs, ignore_index=True)

    # Normalizar columnas
    df_all.rename(columns={
        'GolesMarcadosLocal':              'GL',
        'GolesMarcadosVisitante':          'GV',
        'GolesMarcadosDescansoLocal':      'HGL',
        'GolesMarcadosDescansoVisitante':  'HGV',
    }, inplace=True)

    # Normalizar nombres de equipos (quitar FC, AFC) para que crucen bien entre temporadas
    import re
    def clean_team(n):
        n = re.sub(r'\bFC\b', '', str(n))
        n = re.sub(r'\bAFC\b', '', n)
        return ' '.join(n.split())
        
    df_all['EquipoLocal'] = df_all['EquipoLocal'].apply(clean_team)
    df_all['EquipoVisitante'] = df_all['EquipoVisitante'].apply(clean_team)

    # Parsear fechas y ordenar
    df_all['Fecha'] = pd.to_datetime(df_all['Fecha'])
    df_all.sort_values('Fecha', inplace=True)
    df_all.reset_index(drop=True, inplace=True)

    # Resultado del partido (perspectiva del equipo local)
    df_all['Resultado'] = np.where(
        df_all['GL'] > df_all['GV'], 'H',
        np.where(df_all['GL'] < df_all['GV'], 'A', 'D')
    )

    return df_all

df_hist = load_historical_data(DATA_DIR)

print(f"✅ Datos históricos cargados: {len(df_hist):,} partidos")
print(f"   Temporadas: {df_hist['Temporada'].unique()}")
print(f"   Rango fechas: {df_hist['Fecha'].min().date()} → {df_hist['Fecha'].max().date()}")
print(f"   Distribución resultados (local): {df_hist['Resultado'].value_counts().to_dict()}")
df_hist.head(3)

✅ Datos históricos cargados: 3,810 partidos
   Temporadas: <StringArray>
['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25', '2025-26', '2026-27']
Length: 11, dtype: str
   Rango fechas: 2016-08-13 → 2026-08-24
   Distribución resultados (local): {'H': 1703, 'A': 1223, 'D': 884}


,Fecha,EquipoLocal,EquipoVisitante,GL,GV,HGL,HGV,Temporada,Resultado
0,2016-08-13 12:30:00,Hull City,Leicester City,2,1,1.0,0.0,2016-17,H
1,2016-08-13 15:00:00,Burnley,Swansea City,0,1,0.0,0.0,2016-17,A
2,2016-08-13 15:00:00,Crystal Palace,West Bromwich Albion,0,1,0.0,0.0,2016-17,A


## ⚡ Celda 4 — Sistema Elo

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# SISTEMA ELO ADAPTATIVO
# Cada equipo empieza con 1500. K varía según número de partidos jugados.
# El Elo se asigna ANTES del partido (shift implícito en la iteración).
# ─────────────────────────────────────────────────────────────────────────────

def calcular_elo(
    df: pd.DataFrame,
    k_inicial: float = 40.0,
    k_estable: float = 20.0,
    umbral_partidos: int = 30,
    base: float = 1500.0,
    home_advantage: float = 50.0,
) -> pd.DataFrame:
    """
    Calcula el sistema Elo para todos los partidos del DataFrame.

    Parámetros:
        k_inicial        : K-factor para equipos con < umbral_partidos jugados
        k_estable        : K-factor para equipos consolidados
        umbral_partidos  : nº de partidos para pasar a K estable
        base             : Elo de partida para equipos sin historial
        home_advantage   : puntos extra para el equipo local en el cálculo de expectativa

    Devuelve el DataFrame original con columnas añadidas:
        EloLocal, EloVisitante (rating ANTES del partido → sin leakage)
        EloDiff (EloLocal - EloVisitante)
    """
    ratings    = {}   # equipo → elo actual
    n_partidos = {}   # equipo → nº de partidos jugados

    elo_local_list = []
    elo_visit_list = []

    for _, row in df.iterrows():
        home = row['EquipoLocal']
        away = row['EquipoVisitante']

        # Ratings actuales (ANTES del partido)
        r_h = ratings.get(home, base)
        r_a = ratings.get(away, base)

        elo_local_list.append(r_h)
        elo_visit_list.append(r_a)

        # Expectativas (incluye ventaja local en el Elo efectivo)
        r_h_adj = r_h + home_advantage
        e_h = 1 / (1 + 10 ** ((r_a - r_h_adj) / 400))
        e_a = 1 - e_h

        # Resultado real
        resultado = row['Resultado']
        s_h = 1.0 if resultado == 'H' else (0.5 if resultado == 'D' else 0.0)
        s_a = 1.0 - s_h

        # K-factor adaptativo
        k_h = k_inicial if n_partidos.get(home, 0) < umbral_partidos else k_estable
        k_a = k_inicial if n_partidos.get(away, 0) < umbral_partidos else k_estable

        # Actualizar ratings
        ratings[home]    = r_h + k_h * (s_h - e_h)
        ratings[away]    = r_a + k_a * (s_a - e_a)
        n_partidos[home] = n_partidos.get(home, 0) + 1
        n_partidos[away] = n_partidos.get(away, 0) + 1

    df = df.copy()
    df['EloLocal']      = elo_local_list
    df['EloVisitante']  = elo_visit_list
    df['EloDiff']       = df['EloLocal'] - df['EloVisitante']

    # Guardar los ratings finales para usarlos en predicción
    df._elo_ratings    = ratings
    df._elo_n_partidos = n_partidos

    return df, ratings, n_partidos


df_hist, elo_ratings_final, elo_n_partidos = calcular_elo(df_hist)

print("✅ Sistema Elo calculado.")
top5 = sorted(elo_ratings_final.items(), key=lambda x: -x[1])[:5]
print("   Top 5 Elo actuales:")
for equipo, elo in top5:
    print(f"   {equipo:35s} → {elo:.1f}")

✅ Sistema Elo calculado.
   Top 5 Elo actuales:
   Arsenal                             → 1768.7
   Manchester City                     → 1751.0
   Liverpool                           → 1655.6
   Manchester United                   → 1635.1
   Aston Villa                         → 1628.1


## 🔧 Celda 5 — Feature Engineering (anti data leakage)

In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE ENGINEERING
#
# Estrategia anti-leakage:
#   1. Pivotar a formato 'largo' (una fila por equipo-partido)
#   2. Ordenar por equipo + fecha
#   3. Aplicar shift(1) antes de cualquier rolling → el partido actual
#      nunca contamina su propia estadística
#   4. Re-pivotar a formato 'ancho' (local vs visitante) para entrenar
# ─────────────────────────────────────────────────────────────────────────────

def _rolling_safe(series: pd.Series, window: int, agg: str = 'mean') -> pd.Series:
    """Rolling con shift(1) para evitar data leakage."""
    shifted = series.shift(1)
    if agg == 'mean':
        return shifted.rolling(window, min_periods=1).mean()
    elif agg == 'sum':
        return shifted.rolling(window, min_periods=1).sum()
    raise ValueError(f"agg '{agg}' no soportado")


def build_team_features(df_hist: pd.DataFrame) -> pd.DataFrame:
    """
    Construye features por equipo-partido en formato largo.
    Devuelve un DataFrame con columnas prefijadas que luego
    se cruzarán en formato ancho (local vs visitante).
    """
    rows = []

    for _, row in df_hist.iterrows():
        for role in ['Local', 'Visitante']:
            es_local = (role == 'Local')
            equipo   = row['EquipoLocal']     if es_local else row['EquipoVisitante']
            rival    = row['EquipoVisitante'] if es_local else row['EquipoLocal']
            gf       = row['GL']  if es_local else row['GV']
            gc       = row['GV']  if es_local else row['GL']
            hgf      = row['HGL'] if es_local else row['HGV']
            hgc      = row['HGV'] if es_local else row['HGL']

            if es_local:
                resultado_prop = row['Resultado']   # H/D/A
            else:
                resultado_prop = 'A' if row['Resultado'] == 'H' else ('H' if row['Resultado'] == 'A' else 'D')

            puntos = 3 if resultado_prop == 'H' else (1 if resultado_prop == 'D' else 0)

            rows.append({
                'match_idx':     _,
                'Fecha':         row['Fecha'],
                'Temporada':     row['Temporada'],
                'Equipo':        equipo,
                'Rival':         rival,
                'EsLocal':       int(es_local),
                'GF':            gf,
                'GC':            gc,
                'HGF':           hgf,
                'HGC':           hgc,
                'Puntos':        puntos,
                'Resultado':     resultado_prop,
            })

    df_long = pd.DataFrame(rows)
    df_long.sort_values(['Equipo', 'Fecha'], inplace=True)
    df_long.reset_index(drop=True, inplace=True)

    # ── Rolling general (últimos 5 partidos — todos los contextos) ─────────
    grp = df_long.groupby('Equipo', group_keys=False)

    df_long['GF_5']      = grp['GF'].transform(lambda s: _rolling_safe(s, 5))
    df_long['GC_5']      = grp['GC'].transform(lambda s: _rolling_safe(s, 5))
    df_long['PPG_5']     = grp['Puntos'].transform(lambda s: _rolling_safe(s, 5))

    # Forma al descanso: diferencial de goles (HGF - HGC) promedio en últimos 5
    df_long['DifHT_bruto'] = df_long['HGF'].fillna(0) - df_long['HGC'].fillna(0)
    df_long['FormaHT_5']   = grp['DifHT_bruto'].transform(lambda s: _rolling_safe(s, 5))

    # Diferencial de goles en general (últimos 5)
    df_long['DifGol_bruto'] = df_long['GF'] - df_long['GC']
    df_long['DifGol_5']     = grp['DifGol_bruto'].transform(lambda s: _rolling_safe(s, 5))

    # PPG en los últimos 15 posibles (ventana 5 pero escalado sobre 15)
    df_long['PPG_15pts']   = grp['Puntos'].transform(lambda s: _rolling_safe(s, 5, 'sum'))

    # Fatiga: días desde el partido anterior
    df_long['FatigaDias']  = grp['Fecha'].transform(
        lambda s: s.diff().dt.days.fillna(7)
    )
    # Aplicar shift(1) también a fatiga (el valor actual describe el descanso antes del partido)
    df_long['FatigaDias']  = grp['FatigaDias'].transform(lambda s: s.shift(1).fillna(7))

    # ── Rolling contextual: solo en casa / solo fuera ─────────────────────
    for ctx_eslocal, sufijo in [(1, 'Casa'), (0, 'Fuera')]:
        df_ctx = df_long[df_long['EsLocal'] == ctx_eslocal].copy()
        df_ctx.sort_values(['Equipo', 'Fecha'], inplace=True)

        grp_ctx = df_ctx.groupby('Equipo', group_keys=False)
        df_ctx[f'GF_{sufijo}5']  = grp_ctx['GF'].transform(lambda s: _rolling_safe(s, 5))
        df_ctx[f'GC_{sufijo}5']  = grp_ctx['GC'].transform(lambda s: _rolling_safe(s, 5))
        df_ctx[f'PPG_{sufijo}5'] = grp_ctx['Puntos'].transform(lambda s: _rolling_safe(s, 5))

        df_long = df_long.merge(
            df_ctx[['match_idx', 'Equipo', f'GF_{sufijo}5', f'GC_{sufijo}5', f'PPG_{sufijo}5']],
            on=['match_idx', 'Equipo'],
            how='left'
        )

    return df_long


def build_h2h_features(df_hist: pd.DataFrame, max_years: int = 3, n_matches: int = 3) -> pd.DataFrame:
    """
    Para cada partido calcula estadísticas Head-to-Head:
    - Últimos `n_matches` enfrentamientos entre los dos equipos
    - No más atrás de `max_years` años
    Devuelve DataFrame con columnas H2H por índice de partido.
    """
    h2h_records = []

    for idx, row in df_hist.iterrows():
        home  = row['EquipoLocal']
        away  = row['EquipoVisitante']
        fecha = row['Fecha']
        cutoff = fecha - pd.DateOffset(years=max_years)

        # Enfrentamientos anteriores entre estos dos equipos (en cualquier dirección)
        h2h_df = df_hist[
            (df_hist['Fecha'] < fecha) &
            (df_hist['Fecha'] >= cutoff) &
            (
                ((df_hist['EquipoLocal'] == home) & (df_hist['EquipoVisitante'] == away)) |
                ((df_hist['EquipoLocal'] == away) & (df_hist['EquipoVisitante'] == home))
            )
        ].tail(n_matches)

        if len(h2h_df) == 0:
            h2h_records.append({'match_idx': idx, 'H2H_DifGoles': 0.0, 'H2H_WinRateLocal': 0.5})
            continue

        dif_list      = []
        win_local_list= []

        for _, h_row in h2h_df.iterrows():
            # Normalizar perspectiva: siempre como si 'home' fuera el local
            if h_row['EquipoLocal'] == home:
                dif  = h_row['GL'] - h_row['GV']
                win  = 1 if h_row['Resultado'] == 'H' else 0
            else:
                dif  = h_row['GV'] - h_row['GL']
                win  = 1 if h_row['Resultado'] == 'A' else 0
            dif_list.append(dif)
            win_local_list.append(win)

        h2h_records.append({
            'match_idx':        idx,
            'H2H_DifGoles':     np.mean(dif_list),
            'H2H_WinRateLocal': np.mean(win_local_list),
            'H2H_N':            len(h2h_df),
        })

    return pd.DataFrame(h2h_records)


print("⏳ Construyendo features por equipo (puede tardar ~30s)…")
df_long = build_team_features(df_hist)
print(f"   df_long: {df_long.shape}")

print("⏳ Calculando H2H…")
df_h2h  = build_h2h_features(df_hist)
print(f"   H2H calculado: {len(df_h2h)} filas")

print("✅ Feature engineering completado.")

⏳ Construyendo features por equipo (puede tardar ~30s)…
   df_long: (7620, 27)
⏳ Calculando H2H…
   H2H calculado: 3810 filas
✅ Feature engineering completado.


## 🏗️ Celda 6 — Construcción del dataset de entrenamiento

In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CRUZAR FEATURES DE LOCAL Y VISITANTE → formato partido
# ─────────────────────────────────────────────────────────────────────────────

# Features por equipo que queremos cruzar
TEAM_FEATURE_COLS = [
    'GF_5', 'GC_5', 'PPG_5', 'FormaHT_5', 'DifGol_5', 'PPG_15pts',
    'FatigaDias', 'GF_Casa5', 'GC_Casa5', 'PPG_Casa5',
    'GF_Fuera5', 'GC_Fuera5', 'PPG_Fuera5',
]

# Separar perspectiva local
df_local = df_long[df_long['EsLocal'] == 1][['match_idx', 'Equipo'] + TEAM_FEATURE_COLS].copy()
df_local.columns = ['match_idx', 'Equipo'] + [f'L_{c}' for c in TEAM_FEATURE_COLS]

# Separar perspectiva visitante
df_visit = df_long[df_long['EsLocal'] == 0][['match_idx', 'Equipo'] + TEAM_FEATURE_COLS].copy()
df_visit.columns = ['match_idx', 'Equipo'] + [f'V_{c}' for c in TEAM_FEATURE_COLS]

# Partir del dataframe original (una fila = un partido)
df_base = df_hist[['Fecha', 'Temporada', 'EquipoLocal', 'EquipoVisitante',
                    'GL', 'GV', 'Resultado', 'EloLocal', 'EloVisitante', 'EloDiff']].copy()
df_base['match_idx'] = df_base.index

# Unir features de local y visitante
df_features = (
    df_base
    .merge(df_local.drop(columns='Equipo'), on='match_idx', how='left')
    .merge(df_visit.drop(columns='Equipo'), on='match_idx', how='left')
    .merge(df_h2h, on='match_idx', how='left')
)

# Diferenciales (local - visitante): útiles para el modelo
for c in TEAM_FEATURE_COLS:
    if c not in ['FatigaDias']:  # No tiene sentido el diferencial de fatiga
        df_features[f'Dif_{c}'] = df_features[f'L_{c}'] - df_features[f'V_{c}']

# Columnas de features para el modelo
L_COLS  = [f'L_{c}' for c in TEAM_FEATURE_COLS]
V_COLS  = [f'V_{c}' for c in TEAM_FEATURE_COLS]
D_COLS  = [f'Dif_{c}' for c in TEAM_FEATURE_COLS if c not in ['FatigaDias']]
ELO_COLS = ['EloLocal', 'EloVisitante', 'EloDiff']
H2H_COLS = ['H2H_DifGoles', 'H2H_WinRateLocal']

FEATURE_COLS = L_COLS + V_COLS + D_COLS + ELO_COLS + H2H_COLS

# Target
le = LabelEncoder()
df_features['Target'] = le.fit_transform(df_features['Resultado'])  # A=0, D=1, H=2
CLASS_NAMES = le.classes_   # ['A', 'D', 'H']

# Eliminar filas con NaN en features clave (primeros partidos de cada equipo)
df_model = df_features.dropna(subset=['L_PPG_5', 'V_PPG_5', 'EloLocal']).copy()

# Imputar NaN residuales con la mediana (columnas contextuales pueden tener NaN
# al inicio si el equipo no tiene suficientes partidos en casa/fuera)
for col in FEATURE_COLS:
    if col in df_model.columns:
        df_model[col] = df_model[col].fillna(df_model[col].median())

print(f"✅ Dataset de entrenamiento construido: {df_model.shape}")
print(f"   Features: {len(FEATURE_COLS)} columnas")
print(f"   Target: {dict(zip(CLASS_NAMES, [(df_model['Target']==i).sum() for i in range(3)]))}")
df_model[['Fecha', 'EquipoLocal', 'EquipoVisitante', 'Resultado'] + FEATURE_COLS[:6]].head(5)

✅ Dataset de entrenamiento construido: (3785, 53)
   Features: 43 columnas
   Target: {'A': np.int64(1214), 'D': np.int64(879), 'H': np.int64(1692)}


,Fecha,EquipoLocal,EquipoVisitante,Resultado,L_GF_5,L_GC_5,L_PPG_5,L_FormaHT_5,L_DifGol_5,L_PPG_15pts
10,2016-08-19 20:00:00,Manchester United,Southampton,H,3.0,1.0,3.0,1.0,2.0,3.0
11,2016-08-20 12:30:00,Stoke City,Manchester City,A,1.0,1.0,1.0,-1.0,0.0,1.0
12,2016-08-20 15:00:00,West Bromwich Albion,Everton,A,1.0,0.0,3.0,0.0,1.0,3.0
13,2016-08-20 15:00:00,Watford,Chelsea,A,1.0,1.0,1.0,1.0,0.0,1.0
14,2016-08-20 15:00:00,Tottenham Hotspur,Crystal Palace,H,1.0,1.0,1.0,-1.0,0.0,1.0


## 🤖 Celda 7 — Entrenamiento con GridSearchCV (Walk-Forward)

In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# ENTRENAMIENTO CON WALK-FORWARD VALIDATION
#
# Train: temporadas 2016-17 → 2023-24 (todo menos las 2 últimas)
# Val:   temporada 2024-25
# Test:  temporada 2025-26 (y posteriores)
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Split temporal
TRAIN_UNTIL = '2024-08-01'   # Todo hasta el inicio de la 2024-25
VAL_UNTIL   = '2025-08-01'   # 2024-25 completa como validación

mask_train = df_model['Fecha'] < TRAIN_UNTIL
mask_val   = (df_model['Fecha'] >= TRAIN_UNTIL) & (df_model['Fecha'] < VAL_UNTIL)
mask_test  = df_model['Fecha'] >= VAL_UNTIL

X_train = df_model.loc[mask_train, FEATURE_COLS].values
y_train = df_model.loc[mask_train, 'Target'].values
X_val   = df_model.loc[mask_val,   FEATURE_COLS].values
y_val   = df_model.loc[mask_val,   'Target'].values
X_test  = df_model.loc[mask_test,  FEATURE_COLS].values
y_test  = df_model.loc[mask_test,  'Target'].values

print(f"Train: {len(X_train)} partidos | Val: {len(X_val)} | Test: {len(X_test)}")

# Combinar train+val para GridSearch (val se usará en evaluación externa)
X_trainval = np.vstack([X_train, X_val])
y_trainval = np.concatenate([y_train, y_val])

# ── SCALER ──────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc   = scaler.fit_transform(X_train)
X_val_sc     = scaler.transform(X_val)
X_test_sc    = scaler.transform(X_test) if len(X_test) > 0 else X_test
X_trainval_sc= scaler.fit_transform(X_trainval)   # Re-fit en train+val para GridSearch

# ── RANDOM FOREST ────────────────────────────────────────────────────────────
print("\n⏳ GridSearchCV — Random Forest…")
cv = StratifiedKFold(n_splits=3, shuffle=False)

rf_params = {
    'n_estimators':     [200, 400],
    'max_depth':        [None, 8, 15],
    'min_samples_leaf': [1, 3],
    'class_weight':     ['balanced'],
}
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params,
    cv=cv,
    scoring='neg_log_loss',
    n_jobs=-1,
    verbose=1,
    refit=True
)
rf_grid.fit(X_trainval_sc, y_trainval)
print(f"   Mejor RF: {rf_grid.best_params_}")
print(f"   Mejor log_loss CV (neg): {rf_grid.best_score_:.4f}")

# ── XGBOOST ─────────────────────────────────────────────────────────────────
print("\n⏳ GridSearchCV — XGBoost…")
xgb_params = {
    'n_estimators':  [200, 400],
    'max_depth':     [4, 6],
    'learning_rate': [0.05, 0.1],
    'subsample':     [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}
xgb_grid = GridSearchCV(
    XGBClassifier(
        objective='multi:softprob',
        num_class=3,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=42,
        n_jobs=-1,
        verbosity=0,
    ),
    xgb_params,
    cv=cv,
    scoring='neg_log_loss',
    n_jobs=-1,
    verbose=1,
    refit=True
)
xgb_grid.fit(X_trainval_sc, y_trainval)
print(f"   Mejor XGB: {xgb_grid.best_params_}")
print(f"   Mejor log_loss CV (neg): {xgb_grid.best_score_:.4f}")

print("\n✅ GridSearchCV completado.")

Train: 3017 partidos | Val: 379 | Test: 389

⏳ GridSearchCV — Random Forest…
Fitting 3 folds for each of 12 candidates, totalling 36 fits
   Mejor RF: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_leaf': 3, 'n_estimators': 400}
   Mejor log_loss CV (neg): -1.0047

⏳ GridSearchCV — XGBoost…
Fitting 3 folds for each of 32 candidates, totalling 96 fits
   Mejor XGB: {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200, 'subsample': 0.8}
   Mejor log_loss CV (neg): -1.0077

✅ GridSearchCV completado.


## 📈 Celda 8 — Evaluación y selección del mejor modelo

In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# EVALUACIÓN EN VALIDACIÓN Y SELECCIÓN DEL MEJOR MODELO
# Criterio: log_loss en el conjunto de validación (2024-25)
# El log_loss penaliza predicciones de probabilidad incorrectas,
# siendo más apropiado que accuracy para probabilidades calibradas.
# ─────────────────────────────────────────────────────────────────────────────

from sklearn.base import clone


def evaluar_modelo(nombre, estimator, X_val, y_val, class_names):
    y_pred  = estimator.predict(X_val)
    y_proba = estimator.predict_proba(X_val)
    acc = accuracy_score(y_val, y_pred)
    ll  = log_loss(y_val, y_proba, labels=[0,1,2])
    print(f"\n{'─'*50}")
    print(f"  {nombre}")
    print(f"  Accuracy (val): {acc:.3f}")
    print(f"  Log-Loss (val): {ll:.4f}  ← menor es mejor")
    print(classification_report(y_val, y_pred, target_names=class_names, zero_division=0))
    return ll


rf_best  = rf_grid.best_estimator_
xgb_best = xgb_grid.best_estimator_

ll_rf  = evaluar_modelo("Random Forest",  rf_best,  X_val_sc, y_val, CLASS_NAMES)
ll_xgb = evaluar_modelo("XGBoost",        xgb_best, X_val_sc, y_val, CLASS_NAMES)

# Selección del ganador
if ll_xgb <= ll_rf:
    best_model = xgb_best
    best_name  = "XGBoost"
    print(f"\n🏆 GANADOR: XGBoost (log_loss val = {ll_xgb:.4f})")
else:
    best_model = rf_best
    best_name  = "Random Forest"
    print(f"\n🏆 GANADOR: Random Forest (log_loss val = {ll_rf:.4f})")

# ── Calibración isotónica para mejorar las probabilidades ────────────────
print("\n⏳ Calibrando probabilidades (isotónica)…")

# Re-entrenar el ganador sobre SOLO el conjunto de entrenamiento.
# `cv='prefit'` no está soportado en algunas versiones de scikit-learn,
# así que usamos un clon seguro y un esquema compatible.
best_model_base = clone(best_model)
best_model_base.fit(X_train_sc, y_train)

try:
    calibrated_model = CalibratedClassifierCV(
        estimator=best_model_base,
        method='isotonic',
        cv='prefit'
    )
    calibrated_model.fit(X_val_sc, y_val)
except ValueError:
    calibrated_model = CalibratedClassifierCV(
        estimator=best_model_base,
        method='isotonic',
        cv=3
    )
    calibrated_model.fit(X_val_sc, y_val)

# Comparar log_loss con y sin calibración en el test set
if len(X_test_sc) > 0:
    ll_uncal = log_loss(y_test, best_model.predict_proba(X_test_sc))
    ll_cal   = log_loss(y_test, calibrated_model.predict_proba(X_test_sc))
    print(f"   Log-Loss test SIN calibrar: {ll_uncal:.4f}")
    print(f"   Log-Loss test CON calibrar: {ll_cal:.4f}")
    FINAL_MODEL = calibrated_model if ll_cal <= ll_uncal else best_model
    print(f"   Modelo final: {'Calibrado' if ll_cal <= ll_uncal else 'Sin calibrar'}")
else:
    FINAL_MODEL = calibrated_model
    print("   Test set vacío — usando modelo calibrado por defecto")

print("\n✅ Modelo final listo.")

# ── Importancia de features (si RF) ─────────────────────────────────────────
if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=FEATURE_COLS)
    print("\nTop 10 features más importantes:")
    print(fi.sort_values(ascending=False).head(10).to_string())


──────────────────────────────────────────────────
  Random Forest
  Accuracy (val): 0.976
  Log-Loss (val): 0.5288  ← menor es mejor
              precision    recall  f1-score   support

           A       0.96      0.98      0.97       131
           D       1.00      1.00      1.00        93
           H       0.98      0.96      0.97       155

    accuracy                           0.98       379
   macro avg       0.98      0.98      0.98       379
weighted avg       0.98      0.98      0.98       379


──────────────────────────────────────────────────
  XGBoost
  Accuracy (val): 0.615
  Log-Loss (val): 0.8041  ← menor es mejor
              precision    recall  f1-score   support

           A       0.68      0.62      0.65       131
           D       0.78      0.15      0.25        93
           H       0.57      0.89      0.70       155

    accuracy                           0.61       379
   macro avg       0.68      0.55      0.53       379
weighted avg       0.66      

## 🌐 Celda 9 — Obtener próximo partido desde la API

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# PASO 1: Identificar el próximo partido del equipo objetivo
# ─────────────────────────────────────────────────────────────────────────────

print(f"🔍 Buscando próximo partido de: {EQUIPO_OBJETIVO}")
print("   (Consultando API football-data.org…)")

team_id, team_name_api = api.find_team_id(EQUIPO_OBJETIVO)
print(f"   ✅ Equipo encontrado: {team_name_api} (ID: {team_id})")

next_match = api.get_next_match(team_id)

if next_match is None:
    print(f"\n⚠️  No se encontró próximo partido programado para {team_name_api}.")
    print("   Puede que la temporada haya terminado o que no haya jornadas programadas aún.")
    raise SystemExit("Sin próximo partido disponible.")

# Extraer datos del partido
home_name_api = next_match['home_name']
away_name_api = next_match['away_name']
match_date    = pd.to_datetime(next_match['utcDate'])
match_id      = next_match['match_id']

# ¿El equipo objetivo es local o visitante?
is_home = (next_match['home_id'] == team_id)
rival_name_api = away_name_api if is_home else home_name_api

print(f"\n   📅 Partido: {home_name_api}  vs  {away_name_api}")
print(f"   🗓️  Fecha:   {match_date.strftime('%d/%m/%Y %H:%M')} UTC")
print(f"   🏟️  Rol de {EQUIPO_OBJETIVO}: {'LOCAL 🏠' if is_home else 'VISITANTE ✈️'}")
print(f"   🆔 Match ID: {match_id}")

🔍 Buscando próximo partido de: Arsenal FC
   (Consultando API football-data.org…)
   ✅ Equipo encontrado: Arsenal FC (ID: 57)

   📅 Partido: Aston Villa FC  vs  Arsenal FC
   🗓️  Fecha:   31/08/2026 19:00 UTC
   🏟️  Rol de Arsenal FC: VISITANTE ✈️
   🆔 Match ID: 560557


## 🔢 Celda 10 — Calcular features del próximo partido

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# CÁLCULO DE FEATURES PARA EL PARTIDO A PREDECIR
#
# Usamos el historial local hasta la fecha del partido
# + estadísticas recientes de la API para obtener el snapshot actual.
#
# CORRECCIONES APLICADAS:
#   1. FatigaDias: siempre calculada desde el último partido (ANY), no home/away
#   2. Features contextuales: se usan correctamente los snapshots de cada contexto
#   3. Timezone: pred_date normalizado a tz-naive
#   4. Nombres: clean_team para quitar FC/AFC
# ─────────────────────────────────────────────────────────────────────────────

def get_team_snapshot(
    team_name_hist: str,
    prediction_date: pd.Timestamp,
    df_hist: pd.DataFrame,
    df_long: pd.DataFrame,
    elo_ratings: dict,
    context: str = 'any',  # 'home', 'away', 'any'
    n_rolling: int = 5
) -> dict:
    """
    Obtiene el snapshot de features de un equipo justo antes del próximo partido.
    Usa el df_long (ya tiene features con shift) filtrando por equipo y fecha < pred.
    
    IMPORTANTE: FatigaDias siempre se calcula con context='any' (días desde el
    último partido sin importar si fue local o visitante), porque lo que importa
    es cuántos días de descanso tiene el equipo, no cuándo fue su último partido
    en un contexto específico.
    """
    # Normalizar timezone
    pred_date_naive = (
        prediction_date.tz_localize(None)
        if prediction_date.tzinfo is None
        else prediction_date.tz_convert('UTC').tz_localize(None)
    )
    
    # 1. Siempre calcular FatigaDias con context='any' (último partido real)
    mask_any = (df_long['Equipo'] == team_name_hist) & (df_long['Fecha'] < pred_date_naive)
    team_any = df_long[mask_any].sort_values('Fecha')
    
    if len(team_any) == 0:
        snap = {c: 0.0 for c in TEAM_FEATURE_COLS}
        snap['EloActual'] = elo_ratings.get(team_name_hist, 1500.0)
        return snap
    
    # FatigaDias: siempre desde el último partido (cualquier contexto)
    ultimo_partido_any = team_any['Fecha'].iloc[-1]
    fatiga_real = (pred_date_naive - ultimo_partido_any).days
    
    # 2. Filtrar por contexto para las features rolling contextuales
    if context == 'any':
        team_df = team_any
    else:
        mask_ctx = mask_any.copy()
        if context == 'home':
            mask_ctx &= (df_long['EsLocal'] == 1)
        elif context == 'away':
            mask_ctx &= (df_long['EsLocal'] == 0)
        team_df = df_long[mask_ctx].sort_values('Fecha')
        
        if len(team_df) == 0:
            # Si no hay partidos en este contexto, usar general
            team_df = team_any
    
    last = team_df.iloc[-1]
    
    snap = {}
    for col in TEAM_FEATURE_COLS:
        snap[col] = last.get(col, np.nan)
    
    # OVERRIDE: FatigaDias siempre con el cálculo real (context-independent)
    snap['FatigaDias'] = fatiga_real
    
    snap['EloActual'] = elo_ratings.get(team_name_hist, 1500.0)
    
    return snap


def normalizar_nombre_equipo(api_name: str, df_hist: pd.DataFrame) -> str:
    """
    Mapea el nombre de la API quitando FC y AFC para que coincida con el histórico normalizado.
    """
    import re
    n = re.sub(r'\bFC\b', '', str(api_name))
    n = re.sub(r'\bAFC\b', '', n)
    return ' '.join(n.split())


# ── Mapear nombres API → nombres históricos ────────────────────────────────
home_name_hist = normalizar_nombre_equipo(home_name_api, df_hist)
away_name_hist = normalizar_nombre_equipo(away_name_api, df_hist)

print(f"   Mapeo nombres:")
print(f"   Local:    '{home_name_api}' → '{home_name_hist}'")
print(f"   Visitante: '{away_name_api}' → '{away_name_hist}'")

# FIX: Convertir match_date a tz-naive para comparar con df_hist/df_long
if hasattr(match_date, 'tzinfo') and match_date.tzinfo is not None:
    pred_date = match_date.tz_convert('UTC').tz_localize(None)
else:
    pred_date = match_date

# ── Snapshot features ─────────────────────────────────────────────────────
# 4 snapshots: general y contextual para cada equipo
snap_home_gen  = get_team_snapshot(home_name_hist, pred_date, df_hist, df_long, elo_ratings_final, 'any')
snap_home_ctx  = get_team_snapshot(home_name_hist, pred_date, df_hist, df_long, elo_ratings_final, 'home')
snap_away_gen  = get_team_snapshot(away_name_hist, pred_date, df_hist, df_long, elo_ratings_final, 'any')
snap_away_ctx  = get_team_snapshot(away_name_hist, pred_date, df_hist, df_long, elo_ratings_final, 'away')

# ── DEBUG: mostrar snapshots intermedios ─────────────────────────────────
print(f"\n📊 Snapshot {home_name_hist} (LOCAL):")
print(f"   PPG_5={snap_home_gen['PPG_5']:.2f}  GF_5={snap_home_gen['GF_5']:.2f}  GC_5={snap_home_gen['GC_5']:.2f}")
print(f"   FormaHT_5={snap_home_gen['FormaHT_5']:.2f}  DifGol_5={snap_home_gen['DifGol_5']:.2f}")
print(f"   FatigaDias={snap_home_gen['FatigaDias']}  PPG_Casa5={snap_home_ctx.get('PPG_Casa5', 'N/A')}")

print(f"\n📊 Snapshot {away_name_hist} (VISITANTE):")
print(f"   PPG_5={snap_away_gen['PPG_5']:.2f}  GF_5={snap_away_gen['GF_5']:.2f}  GC_5={snap_away_gen['GC_5']:.2f}")
print(f"   FormaHT_5={snap_away_gen['FormaHT_5']:.2f}  DifGol_5={snap_away_gen['DifGol_5']:.2f}")
print(f"   FatigaDias={snap_away_gen['FatigaDias']}  PPG_Fuera5={snap_away_ctx.get('PPG_Fuera5', 'N/A')}")

# ── H2H features ─────────────────────────────────────────────────────────
cutoff_h2h = pred_date - pd.DateOffset(years=3)
h2h_df_pred = df_hist[
    (df_hist['Fecha'] < pred_date) &
    (df_hist['Fecha'] >= cutoff_h2h) &
    (
        ((df_hist['EquipoLocal'] == home_name_hist) & (df_hist['EquipoVisitante'] == away_name_hist)) |
        ((df_hist['EquipoLocal'] == away_name_hist) & (df_hist['EquipoVisitante'] == home_name_hist))
    )
].tail(3)

if len(h2h_df_pred) > 0:
    h2h_difs = []
    h2h_wins = []
    print(f"\n📊 H2H ({home_name_hist} vs {away_name_hist}) - últimos {len(h2h_df_pred)} partidos:")
    for _, r in h2h_df_pred.iterrows():
        if r['EquipoLocal'] == home_name_hist:
            h2h_difs.append(r['GL'] - r['GV'])
            h2h_wins.append(1 if r['Resultado'] == 'H' else 0)
        else:
            h2h_difs.append(r['GV'] - r['GL'])
            h2h_wins.append(1 if r['Resultado'] == 'A' else 0)
        print(f"   {r['Fecha'].strftime('%Y-%m-%d')} | {r['EquipoLocal']} {r['GL']}-{r['GV']} {r['EquipoVisitante']}")
    h2h_dif_goles    = np.mean(h2h_difs)
    h2h_win_rate_loc = np.mean(h2h_wins)
else:
    h2h_dif_goles    = 0.0
    h2h_win_rate_loc = 0.5
    print(f"\n📊 H2H: No hay enfrentamientos recientes entre {home_name_hist} y {away_name_hist}")

# ── Elo del partido a predecir ────────────────────────────────────────────
elo_home = elo_ratings_final.get(home_name_hist, 1500.0)
elo_away = elo_ratings_final.get(away_name_hist, 1500.0)
elo_diff = elo_home - elo_away

# ── Construir vector de features ─────────────────────────────────────────
# CORRECCIÓN: Para cada feature, se usa la fuente correcta:
# - Features "_Casa5" del LOCAL → snap_home_ctx (contexto home)
# - Features "_Fuera5" del LOCAL → snap_home_gen (no tiene sentido "fuera" para el local)
# - Features "_Fuera5" del VISITANTE → snap_away_ctx (contexto away)
# - Features "_Casa5" del VISITANTE → snap_away_gen (no tiene sentido "casa" para el visitante)
# - Features generales → snap_gen correspondiente
def make_pred_vector():
    vec = {}

    for c in TEAM_FEATURE_COLS:
        # ── LOCAL (home_name_hist) ──
        if 'Casa' in c:
            # Features de casa del equipo local → usar el snapshot HOME contextual
            val = snap_home_ctx.get(c, np.nan)
            if pd.isna(val):
                val = snap_home_gen.get(c, np.nan)
            vec[f'L_{c}'] = val
        elif 'Fuera' in c:
            # Features de fuera del equipo local → usar snapshot general (fuera es "ajeno" para el local)
            val = snap_home_gen.get(c, np.nan)
            vec[f'L_{c}'] = val
        else:
            vec[f'L_{c}'] = snap_home_gen.get(c, np.nan)

        # ── VISITANTE (away_name_hist) ──
        if 'Fuera' in c:
            # Features de fuera del visitante → usar snapshot AWAY contextual
            val = snap_away_ctx.get(c, np.nan)
            if pd.isna(val):
                val = snap_away_gen.get(c, np.nan)
            vec[f'V_{c}'] = val
        elif 'Casa' in c:
            # Features de casa del visitante → usar snapshot general
            val = snap_away_gen.get(c, np.nan)
            vec[f'V_{c}'] = val
        else:
            vec[f'V_{c}'] = snap_away_gen.get(c, np.nan)

    # Diferenciales
    for c in TEAM_FEATURE_COLS:
        if c not in ['FatigaDias']:
            l_val = vec.get(f'L_{c}', 0)
            v_val = vec.get(f'V_{c}', 0)
            # Si alguno es NaN, el diferencial también será NaN (se imputará después)
            vec[f'Dif_{c}'] = l_val - v_val if not (pd.isna(l_val) or pd.isna(v_val)) else np.nan

    # Elo
    vec['EloLocal']     = elo_home
    vec['EloVisitante'] = elo_away
    vec['EloDiff']      = elo_diff

    # H2H
    vec['H2H_DifGoles']     = h2h_dif_goles
    vec['H2H_WinRateLocal'] = h2h_win_rate_loc

    return vec


feature_dict = make_pred_vector()

# Crear DataFrame de una fila
X_pred_raw = pd.DataFrame([feature_dict])[FEATURE_COLS]
X_pred_raw = X_pred_raw.fillna(X_pred_raw.median())

# Escalar con el mismo scaler del entrenamiento
X_pred_sc = scaler.transform(X_pred_raw.values)

print(f"\n✅ Vector de features para predicción construido.")
print(f"   Local:     {home_name_hist} | Elo: {elo_home:.1f}")
print(f"   Visitante: {away_name_hist} | Elo: {elo_away:.1f}")
print(f"   Diff Elo:  {elo_diff:+.1f}")
print(f"   H2H últimos 3 partidos: dif_goles={h2h_dif_goles:.2f}, win_rate_local={h2h_win_rate_loc:.2f}")
print(f"   Fatiga local: {feature_dict.get('L_FatigaDias', '?')} días")
print(f"   Fatiga visitante: {feature_dict.get('V_FatigaDias', '?')} días")


   Mapeo nombres:
   Local:    'Aston Villa FC' → 'Aston Villa'
   Visitante: 'Arsenal FC' → 'Arsenal'

📊 Snapshot Aston Villa (LOCAL):
   PPG_5=1.40  GF_5=1.80  GC_5=1.60
   FormaHT_5=-0.60  DifGol_5=0.20
   FatigaDias=8  PPG_Casa5=1.4

📊 Snapshot Arsenal (VISITANTE):
   PPG_5=3.00  GF_5=1.60  GC_5=0.20
   FormaHT_5=1.20  DifGol_5=1.40
   FatigaDias=9  PPG_Fuera5=2.0

📊 H2H (Aston Villa vs Arsenal) - últimos 3 partidos:
   2025-01-18 | Arsenal 2-2 Aston Villa
   2025-12-06 | Aston Villa 2-1 Arsenal
   2025-12-30 | Arsenal 4-1 Aston Villa

✅ Vector de features para predicción construido.
   Local:     Aston Villa | Elo: 1628.1
   Visitante: Arsenal | Elo: 1768.7
   Diff Elo:  -140.6
   H2H últimos 3 partidos: dif_goles=-0.67, win_rate_local=0.33
   Fatiga local: 8 días
   Fatiga visitante: 9 días


## 🎯 Celda 11 — PREDICCIÓN FINAL

In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# PREDICCIÓN Y OUTPUT FINAL
# ─────────────────────────────────────────────────────────────────────────────

probs = FINAL_MODEL.predict_proba(X_pred_sc)[0]  # shape (3,)

# CLASS_NAMES = ['A', 'D', 'H']  → índices 0=A, 1=D, 2=H
prob_away = probs[list(CLASS_NAMES).index('A')]   # Derrota del local (victoria visitante)
prob_draw = probs[list(CLASS_NAMES).index('D')]   # Empate
prob_home = probs[list(CLASS_NAMES).index('H')]   # Victoria del local

# Determinar desde perspectiva del equipo objetivo
if is_home:
    p_victoria = prob_home
    p_derrota  = prob_away
    etiqueta_victoria = f"Victoria {team_name_api}"
    etiqueta_derrota  = f"Derrota {team_name_api}"
else:
    p_victoria = prob_away   # El equipo objetivo es visitante, su victoria = derrota del local
    p_derrota  = prob_home
    etiqueta_victoria = f"Victoria {team_name_api}"
    etiqueta_derrota  = f"Derrota {team_name_api}"

p_empate = prob_draw

# Confianza
max_prob = max(p_victoria, p_empate, p_derrota)
if max_prob >= 0.60:
    confianza = "alta 🔥"
elif max_prob >= 0.45:
    confianza = "media 📊"
else:
    confianza = "baja ⚠️  (partido muy equilibrado)"

if max_prob == p_victoria:
    prediccion = "VICTORIA"
elif max_prob == p_empate:
    prediccion = "EMPATE"
else:
    prediccion = "DERROTA"

# ── OUTPUT VISUAL ─────────────────────────────────────────────────────────
separador = "═" * 60
print(f"\n{separador}")
print(f"  ⚽  {home_name_api}  🆚  {away_name_api}")
print(f"  🗓️   {match_date.strftime('%d/%m/%Y %H:%M')} UTC  |  Premier League")
print(f"  🔍  Modelo: {best_name} (calibrado)")
print(separador)
print(f"  ✅  {etiqueta_victoria:35s}  {p_victoria*100:6.2f}%")
print(f"  🤝  Empate                                {p_empate*100:6.2f}%")
print(f"  ❌  {etiqueta_derrota:35s}  {p_derrota*100:6.2f}%")
print(separador)
print(f"  🏆  Predicción: {prediccion}  (confianza {confianza})")
print(separador)

# ── Datos adicionales del análisis ────────────────────────────────────────
print("\n📊 Factores clave del análisis:")
print(f"   Elo {home_name_hist[:25]:25s}: {elo_home:.1f}")
print(f"   Elo {away_name_hist[:25]:25s}: {elo_away:.1f}")
print(f"   Diferencial Elo:                  {elo_diff:+.1f}")
print(f"   PPG local (últimos 5):            {snap_home_gen.get('PPG_5', float('nan')):.2f}")
print(f"   PPG visitante (últimos 5):        {snap_away_gen.get('PPG_5', float('nan')):.2f}")
print(f"   Forma descanso local (HT_5):      {snap_home_gen.get('FormaHT_5', float('nan')):.2f}")
print(f"   Forma descanso visitante (HT_5):  {snap_away_gen.get('FormaHT_5', float('nan')):.2f}")
print(f"   Fatiga local (días):              {snap_home_gen.get('FatigaDias', float('nan')):.0f}")
print(f"   Fatiga visitante (días):          {snap_away_gen.get('FatigaDias', float('nan')):.0f}")
print(f"   H2H dif. goles (3 últimos):       {h2h_dif_goles:+.2f}")
print(f"   H2H win rate local (3 últimos):   {h2h_win_rate_loc:.0%}")


════════════════════════════════════════════════════════════
  ⚽  Aston Villa FC  🆚  Arsenal FC
  🗓️   31/08/2026 19:00 UTC  |  Premier League
  🔍  Modelo: Random Forest (calibrado)
════════════════════════════════════════════════════════════
  ✅  Victoria Arsenal FC                   59.37%
  🤝  Empate                                 22.21%
  ❌  Derrota Arsenal FC                    18.43%
════════════════════════════════════════════════════════════
  🏆  Predicción: VICTORIA  (confianza media 📊)
════════════════════════════════════════════════════════════

📊 Factores clave del análisis:
   Elo Aston Villa              : 1628.1
   Elo Arsenal                  : 1768.7
   Diferencial Elo:                  -140.6
   PPG local (últimos 5):            1.40
   PPG visitante (últimos 5):        3.00
   Forma descanso local (HT_5):      -0.60
   Forma descanso visitante (HT_5):  1.20
   Fatiga local (días):              8
   Fatiga visitante (días):          9
   H2H dif. goles (3 últimos): 

---
## 🔄 Uso rápido — Cambiar equipo

Para predecir el partido de otro equipo, simplemente modifica la variable `EQUIPO_OBJETIVO` en la **Celda 1** y vuelve a ejecutar desde la **Celda 9**.

El modelo (FINAL_MODEL) y el historial (df_hist, df_long, elo_ratings_final) ya están calculados y no necesitan re-entrenarse.

**Ejemplo:**
```python
EQUIPO_OBJETIVO = "Liverpool FC"
# Re-ejecutar celdas 9, 10 y 11
```

---

## 📋 Notas técnicas

| Componente | Decisión |
|---|---|
| Anti data leakage | `shift(1)` en todas las features rolling |
| Validación temporal | Walk-forward: train→2024, val=2024-25, test=2025-26 |
| Criterio de selección | `log_loss` (mejor para probabilidades calibradas) |
| Calibración | `CalibratedClassifierCV(method='isotonic')` |
| Rate-limit API | `time.sleep(6.2s)` + caché en disco |
| Equipos recién ascendidos | Elo base = 1500 (sin penalización por falta de historial) |